<h1>Table of Contents<span class="tocSkip"></span></h1>
<div class="toc">
    <ul class="toc-item">
        <li>
            <span><a href="#1.-GridSearchCV" data-toc-modified-id="1.-GridSearchCV-1">
                <span class="toc-item-num">1&nbsp;&nbsp;</span>1. GridSearchCV
            </a></span>
        </li>
        <li>
            <span><a href="#2.-RandomizedSearchCV" data-toc-modified-id="2.-RandomizedSearchCV-2">
                <span class="toc-item-num">2&nbsp;&nbsp;</span>2. RandomizedSearchCV
            </a></span>
        </li>
    </ul>
</div>
<hr>

## 1. GridSearchCV

In [1]:
import pandas as pd

# UCI Heart Disease
url = \
"https://raw.githubusercontent.com/sharmaroshan/Heart-UCI-Dataset/master/heart.csv"
df = pd.read_csv(url)

In [2]:
from sklearn.model_selection import train_test_split

# 입력특성(X)과 레이블(y) 분리
X = df.drop(columns=["target"])   # 입력특성
y = df["target"]                   # 레이블: 심장병 여부 (1=있음, 0=없음)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
print(f"학습 데이터: {X_train.shape}, 평가 데이터: {X_test.shape}")

학습 데이터: (242, 13), 평가 데이터: (61, 13)


In [3]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

# Step 2: GridSearchCV 설정 및 학습
param_grid = {
    "n_estimators" : [50, 100, 200],       # 트리 수
    "max_depth"    : [3, 5, 7, None],       # 트리 최대 깊이
    "min_samples_split": [2, 5, 10],        # 분할 최소 샘플 수
}

# 3 × 4 × 3 = 36가지 조합 × 5-fold = 180회 학습
grid_search = GridSearchCV(
    estimator  = RandomForestClassifier(random_state=42),
    param_grid = param_grid,
    scoring    = "accuracy",
    cv         = 5,
    n_jobs     = 1,
    verbose    = 1,
)

In [4]:
grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 36 candidates, totalling 180 fits


GridSearchCV(cv=5, estimator=RandomForestClassifier(random_state=42), n_jobs=1,
             param_grid={'max_depth': [3, 5, 7, None],
                         'min_samples_split': [2, 5, 10],
                         'n_estimators': [50, 100, 200]},
             scoring='accuracy', verbose=1)

In [5]:
print("최적 파라미터:", grid_search.best_params_)
print("CV 최고 정확도:", round(grid_search.best_score_, 4))

최적 파라미터: {'max_depth': 3, 'min_samples_split': 2, 'n_estimators': 200}
CV 최고 정확도: 0.8434


In [7]:
best_model = grid_search.best_estimator_
test_acc = best_model.score(X_test, y_test)
print("테스트 정확도:", round(test_acc, 4))

테스트 정확도: 0.8033


In [8]:
results = pd.DataFrame(grid_search.cv_results_)
top5 = results[["params","mean_test_score","std_test_score"]]\
           .sort_values("mean_test_score", ascending=False).head(5)
top5["mean_test_score"] = top5["mean_test_score"].round(4)
top5["std_test_score"]  = top5["std_test_score"].round(4)
print(top5.to_string(index=False))

                                                         params  mean_test_score  std_test_score
  {'max_depth': 3, 'min_samples_split': 2, 'n_estimators': 200}           0.8434          0.0429
  {'max_depth': 3, 'min_samples_split': 5, 'n_estimators': 200}           0.8434          0.0429
 {'max_depth': 3, 'min_samples_split': 10, 'n_estimators': 200}           0.8393          0.0461
  {'max_depth': 3, 'min_samples_split': 10, 'n_estimators': 50}           0.8392          0.0427
  {'max_depth': 3, 'min_samples_split': 2, 'n_estimators': 100}           0.8391          0.0342


## 2. RandomizedSearchCV

In [9]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

# 연속형 분포를 포함한 파라미터 공간 정의
param_dist = {
    "n_estimators"      : randint(50, 500),         # 50~499 정수 균등 분포
    "max_depth"         : [3, 5, 7, 10, None],
    "min_samples_split" : randint(2, 20),            # 2~19 정수 균등 분포
    "min_samples_leaf"  : randint(1, 10),
    "max_features"      : ["auto", "sqrt", "log2"], 
}

rand_search = RandomizedSearchCV(
    estimator           = RandomForestClassifier(random_state=42),
    param_distributions = param_dist,
    n_iter              = 50,    # 50가지 조합만 무작위 탐색
    scoring             = "accuracy",
    cv                  = 5,
    random_state        = 42,
    n_jobs              = 1,
    verbose             = 1,
)

In [10]:
rand_search.fit(X_train, y_train)

Fitting 5 folds for each of 50 candidates, totalling 250 fits


RandomizedSearchCV(cv=5, estimator=RandomForestClassifier(random_state=42),
                   n_iter=50, n_jobs=1,
                   param_distributions={'max_depth': [3, 5, 7, 10, None],
                                        'max_features': ['auto', 'sqrt',
                                                         'log2'],
                                        'min_samples_leaf': <scipy.stats._distn_infrastructure.rv_frozen object at 0x000001D7C6994AC8>,
                                        'min_samples_split': <scipy.stats._distn_infrastructure.rv_frozen object at 0x000001D7C68C2AC8>,
                                        'n_estimators': <scipy.stats._distn_infrastructure.rv_frozen object at 0x000001D7C6919908>},
                   random_state=42, scoring='accuracy', verbose=1)

In [11]:
print("최적 파라미터:", rand_search.best_params_)
print("CV 최고 정확도:", round(rand_search.best_score_, 4))

최적 파라미터: {'max_depth': 7, 'max_features': 'sqrt', 'min_samples_leaf': 5, 'min_samples_split': 3, 'n_estimators': 393}
CV 최고 정확도: 0.8557


In [12]:
best_rand = rand_search.best_estimator_
print("테스트 정확도:", round(best_rand.score(X_test, y_test), 4))

테스트 정확도: 0.8197


In [13]:
comparison = {
    "방법"        : ["GridSearchCV", "RandomizedSearchCV"],
    "탐색 횟수"   : [180, 250],
    "CV 정확도"   : [grid_search.best_score_, rand_search.best_score_],
    "테스트 정확도": [best_model.score(X_test,y_test), best_rand.score(X_test,y_test)],
}

pd.DataFrame(comparison)

,방법,탐색 횟수,CV 정확도,테스트 정확도
0,GridSearchCV,180,0.843367,0.803279
1,RandomizedSearchCV,250,0.855697,0.819672
